# 01 — Data Inspection

**Goal:** Load the official working dataset, verify its structure, and confirm it is fit for purpose.

- **Dataset:** `data/processed/ethylene_methane.parquet`
- **Inputs:** `s01`–`s16`
- **Targets:** `methane_ppm`, `ethylene_ppm`
- **Rules:** Preserve temporal order. Do not shuffle. `time_s` is for sorting only, not a model feature.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 42
DATA_PATH = Path('../data/processed/ethylene_methane.parquet')
MEMO_PATH = Path('../results/memos/01_inspection_summary.md')

EXPECTED_SENSORS = [f's{i:02d}' for i in range(1, 17)]
EXPECTED_TARGETS = ['methane_ppm', 'ethylene_ppm']
EXPECTED_TIME    = 'time_s'

## 1. Load Dataset

In [2]:
df = pd.read_parquet(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {df.columns.tolist()}')

Shape: (4178504, 19)
Columns (19): ['time_s', 'methane_ppm', 'ethylene_ppm', 's01', 's02', 's03', 's04', 's05', 's06', 's07', 's08', 's09', 's10', 's11', 's12', 's13', 's14', 's15', 's16']


## 2. Column Verification

In [3]:
cols = set(df.columns)

missing_sensors = [c for c in EXPECTED_SENSORS if c not in cols]
missing_targets = [c for c in EXPECTED_TARGETS if c not in cols]
has_time        = EXPECTED_TIME in cols

print('Missing sensors :', missing_sensors or 'None')
print('Missing targets :', missing_targets or 'None')
print('time_s present  :', has_time)

extra_cols = cols - set(EXPECTED_SENSORS) - set(EXPECTED_TARGETS) - {EXPECTED_TIME}
print('Unexpected cols :', extra_cols or 'None')

Missing sensors : None
Missing targets : None
time_s present  : True
Unexpected cols : None


## 3. Data Types

In [4]:
df.dtypes

time_s          float64
methane_ppm     float64
ethylene_ppm    float64
s01             float64
s02             float64
s03             float64
s04             float64
s05             float64
s06             float64
s07             float64
s08             float64
s09             float64
s10             float64
s11             float64
s12             float64
s13             float64
s14             float64
s15             float64
s16             float64
dtype: object

## 4. Null Check

In [5]:
null_counts = df.isnull().sum()
nulls_present = null_counts[null_counts > 0]
if nulls_present.empty:
    print('No nulls found.')
else:
    print('Columns with nulls:')
    print(nulls_present)

No nulls found.


## 5. Duplicate Rows

In [6]:
n_dupes = df.duplicated().sum()
print(f'Exact duplicate rows: {n_dupes}')

Exact duplicate rows: 0


## 6. Temporal Order Check

In [7]:
if has_time:
    is_monotonic = df[EXPECTED_TIME].is_monotonic_increasing
    n_ties = (df[EXPECTED_TIME].diff() == 0).sum()
    print(f'time_s monotonically increasing : {is_monotonic}')
    print(f'time_s tied values              : {n_ties}')
    print(f'time_s range                    : {df[EXPECTED_TIME].min():.2f} → {df[EXPECTED_TIME].max():.2f}')
else:
    print('time_s not found — cannot verify temporal order.')

time_s monotonically increasing : True
time_s tied values              : 104950
time_s range                    : 0.00 → 41790.19


## 7. Constant or Near-Constant Sensors

In [8]:
sensor_std = df[EXPECTED_SENSORS].std()
constant_sensors = sensor_std[sensor_std == 0].index.tolist()
low_var_sensors  = sensor_std[sensor_std < 1e-6].index.tolist()

print('Constant sensors (std=0)       :', constant_sensors or 'None')
print('Near-constant sensors (std<1e-6):', low_var_sensors or 'None')

Constant sensors (std=0)       : None
Near-constant sensors (std<1e-6): None


## 8. Descriptive Statistics — Sensors

In [9]:
df[EXPECTED_SENSORS].describe().T

,count,mean,std,min,25%,50%,75%,max
s01,4178504.0,2520.156611,253.204455,-56.48,2335.12,2463.49,2676.93,3402.56
s02,4178504.0,1711.449153,118.476246,1568.88,1639.01,1701.30,1754.58,9825.75
s03,4178504.0,2756.596021,1150.494736,-47.78,1581.31,2885.64,3798.99,5567.44
s04,4178504.0,3035.847899,1252.085239,-6.83,1750.76,3199.85,4172.03,6127.68
s05,4178504.0,1863.258476,1104.964817,-12.68,819.76,1393.18,2813.35,4420.84
s06,4178504.0,2386.329001,1425.092114,-41.98,1061.50,1688.83,3605.26,5707.53
s07,4178504.0,2689.913506,1102.780331,-15.28,1533.26,2785.62,3610.69,5304.14
s08,4178504.0,2978.961818,1229.723769,-11.87,1660.44,3136.35,4083.02,5820.37
s09,4178504.0,3541.804377,260.705755,2976.53,3344.63,3481.37,3708.47,4436.43
s10,4178504.0,2823.842425,200.292511,2367.65,2672.74,2782.96,2943.73,3519.34


## 9. Descriptive Statistics — Targets

In [10]:
df[EXPECTED_TARGETS].describe()

,methane_ppm,ethylene_ppm
count,4.178504e+06,4.178504e+06
mean,5.808503e+01,4.369478e+00
std,7.663941e+01,5.521296e+00
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00
75%,1.000000e+02,8.330000e+00
max,2.966700e+02,2.000000e+01


## 10. Target Value Counts (zero vs nonzero)

In [11]:
for target in EXPECTED_TARGETS:
    zero_pct = (df[target] == 0).mean() * 100
    print(f'{target}: {zero_pct:.1f}% zero rows, max={df[target].max():.2f}, min={df[target].min():.2f}')

methane_ppm: 55.7% zero rows, max=296.67, min=0.00
ethylene_ppm: 56.9% zero rows, max=20.00, min=0.00


## 11. Save Inspection Memo

In [12]:
MEMO_PATH.parent.mkdir(parents=True, exist_ok=True)

lines = [
    '# 01 — Data Inspection Summary',
    '',
    f'**Date:** 2026-05-18',
    f'**Dataset:** {DATA_PATH}',
    '',
    '## Shape',
    f'- Rows: {df.shape[0]:,}',
    f'- Columns: {df.shape[1]}',
    '',
    '## Column Check',
    f'- Missing sensors: {missing_sensors or "None"}',
    f'- Missing targets: {missing_targets or "None"}',
    f'- time_s present: {has_time}',
    f'- Unexpected columns: {extra_cols or "None"}',
    '',
    '## Data Quality',
    f'- Null values: {"None" if null_counts.sum() == 0 else nulls_present.to_dict()}',
    f'- Duplicate rows: {n_dupes}',
    f'- Constant sensors: {constant_sensors or "None"}',
    '',
    '## Temporal Order',
    f'- time_s monotonically increasing: {is_monotonic if has_time else "N/A"}',
    f'- time_s ties: {n_ties if has_time else "N/A"}',
    f'- time_s range: {df[EXPECTED_TIME].min():.2f} → {df[EXPECTED_TIME].max():.2f}' if has_time else '- time_s: not found',
    '',
    '## Target Summary',
]

for target in EXPECTED_TARGETS:
    zero_pct = (df[target] == 0).mean() * 100
    lines.append(f'- {target}: min={df[target].min():.2f}, max={df[target].max():.2f}, mean={df[target].mean():.2f}, {zero_pct:.1f}% zero')

lines += [
    '',
    '## Verdict',
    '<!-- Fill in manually after reviewing the output above -->',
    '- [ ] Dataset is fit for EDA — proceed to Step 2',
    '- [ ] Issues found — describe below',
    '',
    '**Notes:**',
]

MEMO_PATH.write_text('\n'.join(lines), encoding='utf-8')
print(f'Memo saved to {MEMO_PATH}')

Memo saved to ..\results\memos\01_inspection_summary.md
